<a href="https://colab.research.google.com/github/Ryan56-56/project1ML/blob/main/bestsofar(5).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import imageio.v2 as imageio
from tqdm import tqdm

# =====================
# Config
# =====================
BATCH_SIZE   = 128
LR           = 0.001
EPOCHS       = 50
WEIGHT_DECAY = 1e-4

# IMPORTANT: dataset copied locally for speed
train_path = "/content/cifar_local/CIFAR-10-images-master/train"
test_path  = "/content/cifar_local/CIFAR-10-images-master/test"

CKPT_DIR = "/content/sample_data/CNN_model_CIFAR10"
os.makedirs(CKPT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [7]:
train_transform = T.Compose([
    T.ToPILImage(),
    T.RandomHorizontalFlip(),
    T.RandomCrop(32, padding=4),
    T.ToTensor(),
    T.Normalize([0.4914, 0.4822, 0.4465],
                [0.2470, 0.2435, 0.2616]),
    T.RandomErasing(p=0.1)
])

test_transform = T.Compose([
    T.ToPILImage(),
    T.ToTensor(),
    T.Normalize([0.4914, 0.4822, 0.4465],
                [0.2470, 0.2435, 0.2616])
])

In [28]:
train_path = "/content/drive/MyDrive/CIFAR-10-images-master/CIFAR-10-images-master/CIFAR-10-images-master/train"
test_path  = "/content/drive/MyDrive/CIFAR-10-images-master/CIFAR-10-images-master/CIFAR-10-images-master/test"

class CIFARFolderDataset(Dataset):
    def __init__(self, root, transform=None):
        self.transform = transform
        self.samples = []
        self.classes = sorted(os.listdir(root))

        for label_idx, folder in enumerate(self.classes):
            folder_path = os.path.join(root, folder)
            for img_name in os.listdir(folder_path):
                self.samples.append((os.path.join(folder_path, img_name), label_idx))

        print(f"Found {len(self.samples)} images in {root}.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = imageio.imread(img_path)

        if self.transform:
            img = self.transform(img)

        return img, label

train_ds = CIFARFolderDataset(train_path, transform=train_transform)
test_ds  = CIFARFolderDataset(test_path,  transform=test_transform)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=0, pin_memory=True, persistent_workers=False)

test_dl = DataLoader(test_ds, batch_size=256, shuffle=False,
                     num_workers=0, pin_memory=True, persistent_workers=False)

Found 50001 images in /content/drive/MyDrive/CIFAR-10-images-master/CIFAR-10-images-master/CIFAR-10-images-master/train.
Found 10000 images in /content/drive/MyDrive/CIFAR-10-images-master/CIFAR-10-images-master/CIFAR-10-images-master/test.


In [9]:
class BetterCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = BetterCNN().to(device)
model = model.to(memory_format=torch.channels_last)
model = torch.compile(model)

In [10]:
opt = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
loss_fn = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler("cuda")

In [41]:
def train_one_epoch(epoch):
    model.train()
    running_loss = 0.0
    loop = tqdm(train_dl, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)

    for xb, yb in loop:
        xb = xb.to(device, non_blocking=True, memory_format=torch.channels_last)
        yb = yb.to(device, non_blocking=True)

        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda"):
            preds = model(xb)
            loss = loss_fn(preds, yb)

        scaler.scale(loss).backward()
        scaler.unscale_(opt) # Explicitly unscale gradients to ensure inf checks are recorded
        scaler.step(opt)
        scaler.update()

        running_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    print(f"Epoch {epoch+1} | Loss: {running_loss/len(train_dl):.4f}")


def evaluate():
    model.eval()
    preds_all, labels_all = [], []

    with torch.no_grad():
        for xb, yb in test_dl:
            xb = xb.to(device, non_blocking=True, memory_format=torch.channels_last)
            preds = model(xb).argmax(1).cpu()

            preds_all.extend(preds.numpy())
            labels_all.extend(yb.numpy())

    acc = accuracy_score(labels_all, preds_all)
    prec = precision_score(labels_all, preds_all, average='weighted', zero_division=0)
    rec = recall_score(labels_all, preds_all, average='weighted', zero_division=0)
    f1 = f1_score(labels_all, preds_all, average='weighted', zero_division=0)

    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1 Score:  {f1:.4f}")

    return f1

In [42]:
resume_path = os.path.join(CKPT_DIR, "last_epoch.pth")
start_epoch = 0

if os.path.exists(resume_path):
    print("Resuming from last checkpoint...")
    checkpoint = torch.load(resume_path)

    # Process the model state_dict to handle '_orig_mod' prefixes
    loaded_state_dict = checkpoint["model"]
    new_state_dict = {}
    for k, v in loaded_state_dict.items():
        if k.startswith("_orig_mod."):
            new_key = k[len("_orig_mod."):]
        else:
            new_key = k
        new_state_dict[new_key] = v

    # Load state dict into the original module (uncompiled) if model is compiled
    if hasattr(model, '_orig_mod'):
        model._orig_mod.load_state_dict(new_state_dict)
    else:
        model.load_state_dict(new_state_dict)

    opt.load_state_dict(checkpoint["optimizer"])
    scheduler.load_state_dict(checkpoint["scheduler"])
    if "scaler" in checkpoint: # Check if scaler state exists in checkpoint
        scaler.load_state_dict(checkpoint["scaler"]) # Load scaler state
    else:
        print("Scaler state not found in checkpoint. Initializing new scaler.")

    start_epoch = checkpoint["epoch"] + 1
    print(f"Resumed at epoch {start_epoch}")

best_f1 = 0.0

for epoch in range(start_epoch, EPOCHS):
    train_one_epoch(epoch)
    scheduler.step()

    # Save the state_dict of the uncompiled model if it's compiled
    model_to_save_state = model
    if hasattr(model, '_orig_mod'):
        model_to_save_state = model._orig_mod

    torch.save({
        "epoch": epoch,
        "model": model_to_save_state.state_dict(),
        "optimizer": opt.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict() # Save scaler state
    }, resume_path)

    print(f"Checkpoint saved: {resume_path}")

Resuming from last checkpoint...
Scaler state not found in checkpoint. Initializing new scaler.
Resumed at epoch 5


RuntimeError: unscale_() has already been called on this optimizer since the last update().

In [55]:
def train_one_epoch(epoch):
    model.train()
    running_loss = 0.0
    loop = tqdm(train_dl, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)

    for xb, yb in loop:
        xb = xb.to(device, non_blocking=True, memory_format=torch.channels_last)
        yb = yb.to(device, non_blocking=True)

        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda"):
            preds = model(xb)
            loss = loss_fn(preds, yb)

        # Standard backward pass without GradScaler
        loss.backward()
        opt.step()

        running_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    avg_loss = running_loss/len(train_dl)
    print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f}")
    # Append the current epoch's average loss to the global list
    global train_losses
    train_losses.append(avg_loss)


resume_path = os.path.join(CKPT_DIR, "last_epoch.pth")
start_epoch = 0
train_losses = [] # Initialize a list to store training losses

if os.path.exists(resume_path):
    print("Resuming from last checkpoint...")
    checkpoint = torch.load(resume_path)

    # Process the model state_dict to handle '_orig_mod' prefixes
    loaded_state_dict = checkpoint["model"]
    new_state_dict = {}
    for k, v in loaded_state_dict.items():
        if k.startswith("_orig_mod."):
            new_key = k[len("_orig_mod."):]
        else:
            new_key = k
        new_state_dict[new_key] = v

    # Load state dict into the original module (uncompiled) if model is compiled
    if hasattr(model, '_orig_mod'):
        model._orig_mod.load_state_dict(new_state_dict)
    else:
        model.load_state_dict(new_state_dict)

    opt.load_state_dict(checkpoint["optimizer"])
    scheduler.load_state_dict(checkpoint["scheduler"])

    start_epoch = checkpoint["epoch"] + 1
    if "train_losses" in checkpoint: # Load previous training losses
        train_losses = checkpoint["train_losses"]
    print(f"Resumed at epoch {start_epoch}")

best_f1 = 0.0

for epoch in range(start_epoch, EPOCHS):
    train_one_epoch(epoch)
    scheduler.step()

    # Save the state_dict of the uncompiled model if it's compiled
    model_to_save_state = model
    if hasattr(model, '_orig_mod'):
        model_to_save_state = model._orig_mod

    torch.save({
        "epoch": epoch,
        "model": model_to_save_state.state_dict(),
        "optimizer": opt.state_dict(),
        "scheduler": scheduler.state_dict(),
        "train_losses": train_losses # Save the accumulated training losses
    }, resume_path)

    print(f"Checkpoint saved: {resume_path}")

Resuming from last checkpoint...
Resumed at epoch 5


Epoch 6 | Loss: 0.9156
Checkpoint saved: /content/sample_data/CNN_model_CIFAR10/last_epoch.pth


Epoch 7 | Loss: 0.9206
Checkpoint saved: /content/sample_data/CNN_model_CIFAR10/last_epoch.pth


Epoch 8 | Loss: 0.9131
Checkpoint saved: /content/sample_data/CNN_model_CIFAR10/last_epoch.pth


Epoch 9 | Loss: 0.9201
Checkpoint saved: /content/sample_data/CNN_model_CIFAR10/last_epoch.pth


...

Epoch 46 | Loss: 0.9177
Checkpoint saved: /content/sample_data/CNN_model_CIFAR10/last_epoch.pth


Epoch 47 | Loss: 0.9182
Checkpoint saved: /content/sample_data/CNN_model_CIFAR10/last_epoch.pth


Epoch 48 | Loss: 0.9165
Checkpoint saved: /content/sample_data/CNN_model_CIFAR10/last_epoch.pth


Epoch 49 | Loss: 0.9142
Checkpoint saved: /content/sample_data/CNN_model_CIFAR10/last_epoch.pth


Epoch 50 | Loss: 0.9232
Checkpoint saved: /content/sample_data/CNN_model_CIFAR10/last_epoch.pth


In [45]:
def train_one_epoch(epoch):
    model.train()
    running_loss = 0.0
    loop = tqdm(train_dl, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)

    for xb, yb in loop:
        xb = xb.to(device, non_blocking=True, memory_format=torch.channels_last)
        yb = yb.to(device, non_blocking=True)

        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda"):
            preds = model(xb)
            loss = loss_fn(preds, yb)

        scaler.scale(loss).backward()
        # The explicit scaler.unscale_(opt) was removed as scaler.step(opt) handles it internally.
        scaler.step(opt)
        scaler.update()

        running_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    avg_loss = running_loss/len(train_dl)
    print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f}")
    # Append the current epoch's average loss to the global list
    global train_losses
    train_losses.append(avg_loss)

def evaluate():
    model.eval()
    preds_all, labels_all = [], []

    with torch.no_grad():
        for xb, yb in test_dl:
            xb = xb.to(device, non_blocking=True, memory_format=torch.channels_last)
            preds = model(xb).argmax(1).cpu()

            preds_all.extend(preds.numpy())
            labels_all.extend(yb.numpy())

    acc = accuracy_score(labels_all, preds_all)
    prec = precision_score(labels_all, preds_all, average='weighted', zero_division=0)
    rec = recall_score(labels_all, preds_all, average='weighted', zero_division=0)
    f1 = f1_score(labels_all, preds_all, average='weighted', zero_division=0)

    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    f1 = f1_score(labels_all, preds_all, average='weighted', zero_division=0)

    print(f"F1 Score:  {f1:.4f}")

    return f1

In [71]:
ckpt_path = "/content/sample_data/CNN_model_CIFAR10/last_epoch.pth"

model = BetterCNN().to(device)
model = model.to(memory_format=torch.channels_last)

# Process the model state_dict to handle '_orig_mod' prefixes
loaded_state_dict = torch.load(ckpt_path)
new_state_dict = {}
for k, v in loaded_state_dict['model'].items(): # Assuming the model state is under 'model' key in the checkpoint
    if k.startswith("_orig_mod."):
        new_key = k[len("_orig_mod."):]
    else:
        new_key = k
    new_state_dict[new_key] = v

model.load_state_dict(new_state_dict)
model = torch.compile(model)

print("Loaded model:", ckpt_path)
print("Running test set evaluation...\n")

evaluate()

Loaded model: /content/sample_data/CNN_model_CIFAR10/last_epoch.pth
Running test set evaluation...

Accuracy:  0.7229
Precision: 0.7276
Recall:    0.7229
F1 Score:  0.7216


0.7215660963548367

In [59]:
import os
import shutil

# The slow Google Drive path
drive_path = "/content/drive/MyDrive/CIFAR-10-images-master/CIFAR-10-images-master/CIFAR-10-images-master"
# The fast local Colab path
local_path = "/content/cifar_local_fast"

# 1. Copy the dataset locally if it hasn't been copied yet
if not os.path.exists(local_path):
    print("Copying files from Drive to local storage... This will take a few minutes.")
    shutil.copytree(drive_path, local_path)
    print("Done copying!")
else:
    print("Files already exist locally!")

# 2. Update the train and test paths to use the new local directory
train_path = os.path.join(local_path, "train")
test_path  = os.path.join(local_path, "test")

# 3. Re-create your datasets and dataloaders with the fast paths
train_ds = CIFARFolderDataset(train_path, transform=train_transform)
test_ds  = CIFARFolderDataset(test_path,  transform=test_transform)

# Note: With local storage, you can safely bump up num_workers for even more speed!
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=2, pin_memory=True)

test_dl = DataLoader(test_ds, batch_size=256, shuffle=False,
                     num_workers=2, pin_memory=True)

print("DataLoaders are successfully updated to use fast local storage!")

Copying files from Drive to local storage... This will take a few minutes.
Done copying!
Found 50001 images in /content/cifar_local_fast/train.
Found 10000 images in /content/cifar_local_fast/test.
DataLoaders are successfully updated to use fast local storage!


KeyboardInterrupt: 